# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What are the most common types of 311 requests in Pittsburgh?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-04-07 13:47:54 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-04-07T13:47:54.452053")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `311 requests service`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 3 datasets matching '311 requests service'

1. **311 Data Archive**
   ID: `311-data`
   **NOTE: THIS DATA STOPPED UPDATING AS OF TUESDAY FEBRUARY 4, 2025** The City of Pittsburgh has transitioned to a new 311 system. We will be working on a new data connection to the
   - 311 Data (CSV) [DataStore] ID: `29462525-62a6-45bf-9b5e-ad2e1c06348d`
   - 311 Data User Guide (HTML) ID: `5601f50b-05c5-4e73-a1a7-bb5cbb54ae3c`
   - 311 Issue and Category Codebook (CSV) [DataStore] ID: `6a2c9de6-9cb8-4da6-bd78-0a913cc5790c`
   - 311 Data (old format, 2015-2022) (CSV) [DataStore] ID: `76fda9d0-69be-4
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": '311 requests service', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Load Data from Resource

**Resource ID:** `29462525-62a6-45bf-9b5e-ad2e1c06348d`
**Limit:** 5

**Result preview:**
```
Resource: 29462525-62a6-45bf-9b5e-ad2e1c06348d
Total records: 815,417
Loaded: 5
Fields (28): group_id, num_requests, parent_closed, status_name, status_code, dept, request_type_name, request_type_id, create_date_et, create_date_utc, last_action_et, last_action_utc, closed_date_et, closed_date_utc, origin, street, cross_street, street_id, cross_street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy

Sample (5 rows):

group_id  num_requests  parent_closed status_name  status_code                 dept request_type_name request_type_id  
```


In [ ]:
# Step 2: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '29462525-62a6-45bf-9b5e-ad2e1c06348d', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 3: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 926,130
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status     case_owner     subject subject_code                   legacy_subject     created_date_et    created_date_utc last_modified_date_et last
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query

**SQL:**
```sql
SELECT "request_type_name", COUNT(*) as count FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d" GROUP BY "request_type_name" ORDER BY count DESC LIMIT 15
```

**Result preview:**
```
SQL: SELECT "request_type_name", COUNT(*) as count FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d" GROUP BY "request_type_name" ORDER BY count DESC LIMIT 15
Rows: 15
Columns: request_type_name, count

                   request_type_name  count
                        Weeds/Debris  77513
                            Potholes  67502
               Missed Refuse Pick Up  42185
                    Snow/Ice removal  34942
                Building Maintenance  28863
                   Refuse Violations  26598
Abandoned Vehicle (parked on street)  23768
                     Illegal Parking  21289
       
```


In [ ]:
# Step 4: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "request_type_name", COUNT(*) as count FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d" GROUP BY "request_type_name" ORDER BY count DESC LIMIT 15'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 5: SQL Analysis Query

**SQL:**
```sql
SELECT "subject", COUNT(*) as count FROM "5202679a-d243-402e-b82a-63189995a942" GROUP BY "subject" ORDER BY count DESC LIMIT 15
```

**Result preview:**
```
SQL: SELECT "subject", COUNT(*) as count FROM "5202679a-d243-402e-b82a-63189995a942" GROUP BY "subject" ORDER BY count DESC LIMIT 15
Rows: 15
Columns: subject, count

                             subject  count
                     Weeds or Debris  91255
                            Potholes  81775
                      Missed Pick Up  78252
                    Snow/Ice Removal  56480
      Refuse or Recycling Violations  47964
         Building Maintenance Issues  34342
Abandoned Vehicle (Parked on Street)  30937
                     Illegal Parking  28028
               Street Light Concerns 
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "subject", COUNT(*) as count FROM "5202679a-d243-402e-b82a-63189995a942" GROUP BY "subject" ORDER BY count DESC LIMIT 15'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here's a comprehensive breakdown of the most common 311 request types in Pittsburgh, drawing from **both** the archive system (through Feb 2025) and the new system (from Feb 2025 onward):

---

## 🏆 Top 311 Request Types in Pittsburgh

Both datasets tell a remarkably consistent story. Here are the top categories, combining data from **over 1.7 million total requests**:

| Rank | Request Type | Archive Count | New System Count |
|------|-------------|--------------|-----------------|
| 1 | **Weeds / Debris** | 77,513 | 91,255 |
| 2 | **Potholes** | 67,502 | 81,775 |
| 3 | **Missed Refuse / Recycling Pick Up** | 60,172 | 78,252 |
| 4 | **Snow / Ice Removal** | 34,942 | 56,480 |
| 5 | **Building Maintenance** | 28,863 | 34,342 |
| 6 | **Refuse/Recycling Violations** | 26,598 | 47,964 |
| 7 | **Abandoned Vehicles** | 23,768 | 30,937 |
| 8 | **Illegal Parking** | 21,289 | 28,028 |
| 9 | **Street Light Repair** | 17,903 | 23,833 |
| 10 | **Litter on Public Property** | 17,852 | 20,611 |

---

### 🔑 Key Takeaways

1. **Weeds/Debris & Potholes dominate** — These two categories alone account for roughly **18–20% of all requests**, reflecting Pittsburgh's aging infrastructure and dense tree canopy.
2. **Missed trash/recycling pickups** rank #3, highlighting ongoing solid waste service concerns.
3. **Snow/Ice removal** spikes seasonally and is a perennial top-5 issue given Pittsburgh winters.
4. **The top 10 are strikingly consistent** across both the old and new 311 systems, showing these are long-standing city priorities.

**Sources:**
- [311 Data Archive](https://data.wprdc.org/dataset/311-data) (815,417 records, up to Feb 2025)
- [Pittsburgh 311 Data](https://data.wprdc.org/dataset/pittsburgh-311-data) (926,130 records, new system from Feb 2025)

> ⚠️ **Note:** The new system uses slightly different category names and includes some historical records imported from the old system, so counts may overlap slightly. Both datasets are available on the **Western PA Regional Data Center (WPRDC)**.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-04-07

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-04-07 13:47:54
- **Query**: What are the most common types of 311 requests in Pittsburgh?
- **Data Source**: WPRDC

---

### 💬 Need Help?

- **Documentation**: [Data Concierge Guide](https://github.com/dathere/data-concierge)
- **Issues**: Report problems or request features on GitHub
- **Extend**: Add your own analysis cells below this one!

---

*Generated by  AI Data Concierge v0.1.0*
